In [ ]:
# imports
import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import numpy as np
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# EDA

In [ ]:
# Read census-bureau headers
with open("../data/census-bureau.columns") as f:
    columns = f.read()
columns = columns.split('\n')[:-1]

df = pd.read_csv("../data/census-bureau.data", names=columns)
df.head()


In [ ]:
# checking the data skewness
label_counts = df["label"].value_counts(normalize=True)
print(label_counts)
print(f"\nClass imbalance ratio: {label_counts.iloc[0]/label_counts.iloc[1]:.2f}:1\n")
# checking null values
print(df.isnull().sum().sort_values(ascending=False))

# since only hispanic origin column has null values, we can fill them with it's own category
df["hispanic origin"] = df["hispanic origin"].fillna("Unknown")

In [ ]:
print("--- Duplicates ---")
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes/len(df)*100:.2f}%)")

print("\n--- Missing values (NaN or '?') ---")
missing_mask = df.isnull() | (df == '?')
missing = pd.DataFrame({
    "count":   missing_mask.sum(),
    "percent": (missing_mask.sum() / len(df) * 100).round(2),
}).query("count > 0").sort_values("percent", ascending=False)
print(missing)

print("\n--- 'Not in universe' frequencies ---")
niu_cols = {}
for col in df.select_dtypes("object").columns:
    n = df[col].str.lower().str.contains("not in universe", na=False).sum()
    if n > 0:
        niu_cols[col] = n
niu_df = pd.Series(niu_cols).sort_values(ascending=False)
print(niu_df)

In [ ]:
# checking data types
print("data types:", df.dtypes)

# int columns that are actually categorical
cat_int_cols = [
        'detailed industry recode',
        'detailed occupation recode',
        "own business or self employed",
        "veterans benefits",
    ]

# Convert to categorical (so TargetEncoder handles them)
for col in cat_int_cols:
    df[col] = df[col].astype(str)

# separating into categorical and numerical columns
categorical_cols = df.select_dtypes(include=['str']).columns.to_list()
numerical_cols = df.select_dtypes(exclude=['str']).columns.to_list()

# remove target and weight columns
categorical_cols = [c for c in categorical_cols if c not in ["label"]]
numerical_cols = [c for c in numerical_cols if c not in ["weight"]]

print("Num categorical columns:", len(categorical_cols), "Num numerical columns:", len(numerical_cols))

In [ ]:
# rename label
df["label_binary"] = df["label"].apply(lambda x: 1 if "50000+" in x else 0)

In [ ]:
# At n~200,000 all p-values will be near zero due to high statistical power
# The U-statistic is the meaningful ranking signal at this sample size
print("\n--- Mann-Whitney U test ---")
CONTINUOUS = [
    "age", "wage per hour", "capital gains", "capital losses",
    "dividends from stocks"
]
mw_results = {}
for col in CONTINUOUS:
    grp0 = df.loc[df["label_binary"] == 0, col].dropna()
    grp1 = df.loc[df["label_binary"] == 1, col].dropna()
    stat, p = mannwhitneyu(grp0, grp1, alternative="two-sided")
    mw_results[col] = {"U-statistic": round(stat, 0), "p-value": p}

mw_df = pd.DataFrame(mw_results).T.sort_values("U-statistic")
mw_df["p-value"] = mw_df["p-value"].apply(lambda x: f"{x:.2e}")
print(mw_df)

In [ ]:
# Chi-square test + Cramér's V for each categorical vs. label
CATEGORICAL = [c for c in df.columns if c not in numerical_cols + ["label", "label_binary", "weight"]]

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

print("\n--- Cramér's V and chi-square p-value vs. income label ---")
assoc_results = {}
for col in CATEGORICAL:
    valid = df[[col, "label"]].dropna()
    try:
        ct = pd.crosstab(valid[col], valid["label"])
        chi2, p, _, _ = chi2_contingency(ct)
        v = cramers_v(valid[col], valid["label"])
        assoc_results[col] = {"Cramér's V": round(v, 4), "p-value": round(p, 6)}
    except Exception:
        pass

assoc_df = pd.DataFrame(assoc_results).T.sort_values("Cramér's V", ascending=False)
print(assoc_df.head(10))

# Model Experimentation

In [ ]:
X = df.drop(columns=['label', 'label_binary', 'weight'])
y = df['label_binary']
weights = df['weight']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights,
    test_size=0.3,
    stratify=y,
    random_state=42
)

In [ ]:
# preprocess and encode variables
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", TargetEncoder(), categorical_cols) # Ordinal encoder was marginally worse, not shown in code
    ]
)

In [ ]:
# fit on train, transform both train and test
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
# train logistic regression
log_reg = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train_processed, y_train, sample_weight=w_train)

In [ ]:
# logistic regression evaluation
y_pred = log_reg.predict(X_test_processed)
y_proba = log_reg.predict_proba(X_test_processed)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:",
      roc_auc_score(y_test, y_proba, sample_weight=w_test))

In [ ]:
# try random forest
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1
)

rf_clf.fit(X_train_processed, y_train, sample_weight=w_train)

In [ ]:
y_pred_rf = rf_clf.predict(X_test_processed)
y_proba_rf = rf_clf.predict_proba(X_test_processed)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:",
      roc_auc_score(y_test, y_proba_rf, sample_weight=w_test))

In [ ]:
# perform grid search for vanilla xgboost (should take around 5 mins)
param_grid = {
    'max_depth': [8, 10, 12],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 400, 600]
}

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(
        scale_pos_weight=sum(y_train == 0) / sum(y_train == 1),
        tree_method='hist',
        random_state=42
    ),
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(
    X_train_processed,
    y_train,
    sample_weight=w_train,
    verbose=False
)

print("Best Hyperparameters:")
print(xgb_grid.best_params_)
print(f"\nBest ROC-AUC Score: {xgb_grid.best_score_:.4f}")

# Evaluate on test set
y_proba_best = xgb_grid.best_estimator_.predict_proba(X_test_processed)[:, 1]
test_auc = roc_auc_score(y_test, y_proba_best, sample_weight=w_test)
print(f"Test ROC-AUC: {test_auc:.4f}")
y_pred_xgb = xgb_grid.best_estimator_.predict(X_test_processed)
y_proba_xgb = xgb_grid.best_estimator_.predict_proba(X_test_processed)[:, 1]

print(classification_report(y_test, y_pred_xgb))

In [ ]:
preprocessor.fit(X_train, y_train)

feature_names = preprocessor.get_feature_names_out()
importances = xgb_grid.best_estimator_.get_booster().get_score(importance_type='weight')

# Create the dataframe from the importances dict
feat_importance_xgb = pd.DataFrame(
    list(importances.items()), columns=['feature', 'importance']
).sort_values('importance', ascending=False)

feat_importance_xgb['column_name'] = feat_importance_xgb['feature'].apply(
    lambda f: feature_names[int(f[1:])] if f.startswith('f') and f[1:].isdigit() else f
)

print("Top 20 Most Important Features (XGBoost):")
print(feat_importance_xgb.head(20)[['column_name', 'importance']])

plt.figure(figsize=(10, 6))
plt.barh(feat_importance_xgb.head(15)['column_name'], feat_importance_xgb.head(15)['importance'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

In [ ]:
# Try SMOTE oversampling on train set and retrain XGBoost
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

In [ ]:
xgb_smote = xgb.XGBClassifier(
    learning_rate=0.01,
    max_depth=8,
    n_estimators=600,
    tree_method='hist',
    random_state=42
)

xgb_smote.fit(X_train_smote, y_train_smote)

y_proba_smote = xgb_smote.predict_proba(X_test_processed)[:, 1]
y_pred_smote  = xgb_smote.predict(X_test_processed)

smote_auc = roc_auc_score(y_test, y_proba_smote, sample_weight=w_test)
print(f"SMOTE XGB Test ROC-AUC: {smote_auc:.4f}")
print(classification_report(y_test, y_pred_smote, sample_weight=w_test, digits=3))

In [ ]:
# training a simple MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=256,
    max_iter=50,
    random_state=42
)

mlp.fit(X_train_processed, y_train)

In [ ]:
y_pred_mlp = mlp.predict(X_test_processed)
y_proba_mlp = mlp.predict_proba(X_test_processed)[:, 1]

print(classification_report(y_test, y_pred_mlp))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_mlp))